# FunctorFlow KET Demo

This notebook is the first concrete bridge from the tutorial-library layer to a
demonstration KET build path. It stays deliberately small: we build a typed KET
block, inspect its ports and IR, and run a toy aggregation example end to end.


In [ ]:
from pathlib import Path
import sys

def _bootstrap_functorflow() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        package_root = candidate / "FunctorFlow"
        if (package_root / "__init__.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError(
        "Could not find the FunctorFlow package. Run this notebook from the repo "
        "root or from FunctorFlow/notebooks."
    )

REPO_ROOT = _bootstrap_functorflow()
print(f"FunctorFlow repo root: {REPO_ROOT}")


In [ ]:
from FunctorFlow import KETBlockConfig, compile_to_callable, get_tutorial_library, ket_block

foundations = get_tutorial_library("foundations")
foundations.name, foundations.macro_names


## Build A KET Block

We parameterize the block explicitly so the notebook mirrors the FunctorFlow
language surface, not an opaque helper API.


In [ ]:
ket = ket_block(
    KETBlockConfig(
        name="TutorialKET",
        source_object="EdgeValues",
        relation_object="TokenIncidence",
        target_object="TokenStates",
        aggregate_name="gather",
        reducer="sum",
    )
)

print(ket.summary())
{name: (port.ref, port.port_type, port.direction) for name, port in ket.ports.items()}


## Run The FunctorFlow Diagram

This is the smallest executable KET-style demonstration we currently have in
FunctorFlow itself.


In [ ]:
compiled = compile_to_callable(ket)
result = compiled.run(
    {
        "EdgeValues": {
            0: 1.0,
            1: 3.0,
            2: 5.0,
        },
        "TokenIncidence": {
            "ctx_a": [0, 1],
            "ctx_b": [1, 2],
        },
    }
)

result.values[ket.port("output")]


## Toward A Real Demonstration Model

The next step after this notebook is to replace the toy Python dictionaries with
a real tensor-backed value flow and start lowering a small KET model through the
FunctorFlow compiler surface.
